# MedTrack_DV – Milestone 1: Data Collection and Preparation
This notebook audits and cleans the patient admission and PMC infrastructure datasets. They remain separate because they do not share a reliable facility key.

In [ ]:
import pandas as pd
from pathlib import Path

patient_path = Path('../data/raw/patient_admissions_raw.csv')
infrastructure_path = Path('../data/raw/pmc_infrastructure_raw.csv')
patient = pd.read_csv(patient_path)
infrastructure = pd.read_csv(infrastructure_path)
print('Patient dataset:', patient.shape)
print('Infrastructure dataset:', infrastructure.shape)

## 1. Initial data quality audit

In [ ]:
def audit(df, name):
    print(f'--- {name} ---')
    print('Rows, columns:', df.shape)
    print('Exact duplicate rows:', df.duplicated().sum())
    print('Missing values by column:')
    display(df.isna().sum().to_frame('Missing'))

audit(patient, 'Patient admissions')
audit(infrastructure, 'PMC infrastructure')

## 2. Clean patient admission records

In [ ]:
p = patient.copy()
p.columns = p.columns.str.strip()
for col in p.select_dtypes(include='object').columns:
    p[col] = p[col].astype('string').str.strip()
p['Date of Admission'] = pd.to_datetime(p['Date of Admission'], errors='coerce')
p['Discharge Date'] = pd.to_datetime(p['Discharge Date'], errors='coerce')
p['Age'] = pd.to_numeric(p['Age'], errors='coerce')
p['Billing Amount'] = pd.to_numeric(p['Billing Amount'], errors='coerce')
p['Length of Stay (Days)'] = (p['Discharge Date'] - p['Date of Admission']).dt.days
removed_duplicates = p.duplicated().sum()
p = p.drop_duplicates().copy()
p['Admission Month'] = p['Date of Admission'].dt.to_period('M').astype('string')
p['Admission Year'] = p['Date of Admission'].dt.year.astype('Int64')
print('Duplicate rows removed:', removed_duplicates)
print('Clean patient shape:', p.shape)
p.head()

## 3. Clean PMC infrastructure records

In [ ]:
h = infrastructure.copy()
h.columns = h.columns.str.strip()
for col in h.select_dtypes(include='object').columns:
    h[col] = h[col].astype('string').str.strip()
for col in ['Class : (Public / Private)', 'Pharmacy Available : Yes/No', 'Ambulance Service Available']:
    if col in h.columns:
        h[col] = h[col].str.title()
for col in ['Pharmacy Available : Yes/No', 'Ambulance Service Available']:
    if col in h.columns:
        h[col] = h[col].replace({'N.A.': 'Not Specified', 'Na': 'Not Specified'})
if 'Ward No.' in h.columns and h['Ward No.'].isna().all():
    h = h.drop(columns=['Ward No.'])
numeric_cols = ['Zone No.', 'Number of Beds in Emergency Wards', 'Number of Beds in facility type', 'Number of Doctors / Physicians', 'Number of Nurses', 'Number of Midwives Professional', 'Average Monthly Patient Footfall', 'Count of Ambulance']
for col in numeric_cols:
    if col in h.columns:
        h[col] = pd.to_numeric(h[col].replace({'N.A.': pd.NA, 'NA': pd.NA, '': pd.NA}), errors='coerce')
for col in ['Zone Name', 'Ward Name']:
    if col in h.columns:
        h[col] = h[col].fillna('Unknown')
print('Clean infrastructure shape:', h.shape)
h.head()

## 4. Export Tableau-ready files

In [ ]:
out = Path('../data/processed')
out.mkdir(parents=True, exist_ok=True)
p.to_csv(out / 'patient_admissions_cleaned.csv', index=False)
h.to_csv(out / 'pmc_infrastructure_cleaned.csv', index=False)
print('Exported both cleaned datasets.')

## 5. Data limitations to document
- The patient admissions data and PMC infrastructure data do not share a reliable hospital identifier; do not join records row by row.
- The patient file has no persistent patient ID suitable for verified readmission tracking.
- The infrastructure file is a facility snapshot, not a time series of occupied beds; true occupancy rate cannot be calculated.
- The provided fields do not include a department identifier or equipment inventory. Department and equipment KPIs need additional source data or must be explicitly marked unavailable.